In [1]:
from __future__ import annotations
from pathlib import Path
import sys
import numpy as np
import pandas as pd


In [2]:
def validate_transaction_data(df: pd.DataFrame, required_columns: Optional[List[str]] = None) -> None:
    """Validate transaction data structure."""
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input data must be a pandas DataFrame.")
    if df.empty:
        raise ValueError("Transaction dataset is empty.")
    columns = required_columns or REQUIRED_COLUMNS
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {', '.join(missing)}")


In [3]:
def load_transactions(file_path: str) -> pd.DataFrame:
    """Load transaction data from CSV."""
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Transaction file not found: {file_path}") from exc
    except Exception as exc:
        raise ValueError(f"Unable to read transaction file: {exc}") from exc
    validate_transaction_data(df)
    return df


In [4]:
transactions =  pd.read_csv("data\\raw_transactions.csv")

In [5]:
REQUIRED_COLUMNS = [
    "TransactionID", "CustomerID", "TransactionDate",
    "ProductID", "Category", "Quantity", "UnitPrice"
]

In [6]:
def clean_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """Clean transaction records and calculate transaction value."""
    validate_transaction_data(df)
    cleaned = df.drop_duplicates().copy()
    cleaned["TransactionDate"] = pd.to_datetime(cleaned["TransactionDate"], errors="coerce")
    for c in ["Quantity", "UnitPrice"]:
        cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce")
    for c in ["CustomerID", "ProductID", "Category"]:
        cleaned[c] = cleaned[c].astype("string").str.strip()
    cleaned = cleaned.dropna(subset=[
        "CustomerID", "TransactionDate", "ProductID", "Category",
        "Quantity", "UnitPrice"
    ])
    cleaned = cleaned[(cleaned["Quantity"] > 0) & (cleaned["UnitPrice"] > 0)]
    cleaned["TotalAmount"] = cleaned["Quantity"] * cleaned["UnitPrice"]
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan).dropna(subset=["TotalAmount"])
    if cleaned.empty:
        raise ValueError("No valid transaction records remain after cleaning.")
    return cleaned.reset_index(drop=True)
    

In [7]:
transactions = clean_transactions(transactions)


In [8]:
def calculate_average_purchase_gap(customer_transactions: pd.DataFrame) -> float:
    """Calculate the mean number of days between purchases."""
    dates = pd.to_datetime(customer_transactions["TransactionDate"], errors="coerce").dropna().sort_values()
    if len(dates) < 2:
        return 0.0
    gaps = dates.diff().dt.total_seconds() / 86400
    return float(gaps.dropna().mean())


In [9]:
def build_customer_features(
    transactions: pd.DataFrame,
    reference_date: pd.Timestamp | None = None,
) -> pd.DataFrame:
    """Aggregate transactions into RFM and behavioral customer features."""
    required = {
        "CustomerID", "TransactionDate", "ProductID", "Category",
        "Quantity", "TotalAmount"
    }
    missing = required - set(transactions.columns)
    if missing:
        raise ValueError(f"Missing columns for feature engineering: {missing}")
    if transactions.empty:
        raise ValueError("Cannot engineer features from empty data.")
    df = transactions.copy()
    df["TransactionDate"] = pd.to_datetime(df["TransactionDate"], errors="coerce")
    df = df.dropna(subset=["CustomerID", "TransactionDate", "TotalAmount"])
    if reference_date is None:
        reference_date = df["TransactionDate"].max() + pd.Timedelta(days=1)

    grouped = df.groupby("CustomerID")
    rfm = grouped.agg(
        LastPurchaseDate=("TransactionDate", "max"),
        Frequency=("TransactionID", "nunique"),
        MonetaryValue=("TotalAmount", "sum"),
    )
    rfm["Recency"] = (reference_date - rfm["LastPurchaseDate"]).dt.days

    behavioral = grouped.agg(
        TotalQuantityPurchased=("Quantity", "sum"),
        UniqueProducts=("ProductID", "nunique"),
        UniqueCategories=("Category", "nunique"),
    )
    behavioral["AOV"] = rfm["MonetaryValue"] / rfm["Frequency"].replace(0, np.nan)

    purchase_gaps = (
        df.groupby("CustomerID")
        .apply(calculate_average_purchase_gap, include_groups=False)
        .rename("AverageDaysBetweenPurchases")
    )

    features = pd.concat(
        [rfm.drop(columns=["LastPurchaseDate"]), behavioral, purchase_gaps],
        axis=1,
    )
    numeric = [
        "Recency", "Frequency", "MonetaryValue", "AOV",
        "TotalQuantityPurchased", "UniqueProducts",
        "UniqueCategories", "AverageDaysBetweenPurchases"
    ]
    for c in numeric:
        features[c] = pd.to_numeric(features[c], errors="coerce")
    features = features.replace([np.inf, -np.inf], np.nan)
    features[numeric] = features[numeric].fillna(0)
    return features.reset_index()

In [10]:
customer_features = build_customer_features(transactions)

In [11]:
customer_features.head(10)

,CustomerID,Frequency,MonetaryValue,Recency,TotalQuantityPurchased,UniqueProducts,UniqueCategories,AOV,AverageDaysBetweenPurchases
0,CUST_0001,4,173.58,24,8,4,2,43.3950,126.000000
1,CUST_0002,3,389.01,5,11,3,2,129.6700,262.500000
2,CUST_0003,4,902.00,114,12,4,2,225.5000,51.666667
3,CUST_0004,1,28.68,362,1,1,1,28.6800,0.000000
4,CUST_0005,4,771.03,14,14,4,4,192.7575,51.333333
5,CUST_0006,2,275.65,113,3,2,2,137.8250,11.000000
6,CUST_0007,2,111.42,162,4,1,2,55.7100,24.000000
7,CUST_0008,1,53.90,390,2,1,1,53.9000,0.000000
8,CUST_0009,2,429.42,13,8,2,2,214.7100,18.000000
9,CUST_0010,2,268.21,31,7,2,2,134.1050,24.000000


In [12]:
def get_model_features(customer_features: pd.DataFrame) -> List[str]:
    """Return behavioral features used by the clustering model."""
    columns = [
        "Recency", "Frequency", "MonetaryValue", "AOV",
        "TotalQuantityPurchased", "UniqueProducts",
        "UniqueCategories", "AverageDaysBetweenPurchases"
    ]
    missing = [c for c in columns if c not in customer_features.columns]
    if missing:
        raise ValueError(f"Missing model features: {missing}")
    return columns

In [13]:
model_features = get_model_features(customer_features)


In [14]:
model_features

['Recency',
 'Frequency',
 'MonetaryValue',
 'AOV',
 'TotalQuantityPurchased',
 'UniqueProducts',
 'UniqueCategories',
 'AverageDaysBetweenPurchases']

In [15]:
def cap_outliers_iqr(df: pd.DataFrame, columns: List[str], multiplier: float = 1.5) -> pd.DataFrame:
    """Cap numerical outliers using IQR boundaries."""
    result = df.copy()
    for column in columns:
        if column not in result.columns:
            continue
        values = pd.to_numeric(result[column], errors="coerce")
        q1, q3 = values.quantile(.25), values.quantile(.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue
        result[column] = values.clip(q1 - multiplier * iqr, q3 + multiplier * iqr)
    return result

def log_transform_features(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """Apply log1p to non-negative skewed features."""
    result = df.copy()
    for column in columns:
        if column in result.columns:
            values = pd.to_numeric(result[column], errors="coerce")
            result[column] = np.log1p(np.clip(values, 0, None))
    return result


In [16]:
def prepare_customer_features(
    customer_df: pd.DataFrame,
    log_features: Optional[List[str]] = None,
    outlier_features: Optional[List[str]] = None,
) -> pd.DataFrame:
    """Impute missing values, cap outliers and log-transform features."""
    if customer_df.empty:
        raise ValueError("Customer feature dataset is empty.")
    result = customer_df.copy()
    features = [
        c for c in result.columns
        if c != "CustomerID" and pd.api.types.is_numeric_dtype(result[c])
    ]
    result[features] = result[features].replace([np.inf, -np.inf], np.nan)
    for c in features:
        result[c] = result[c].fillna(result[c].median() if not pd.isna(result[c].median()) else 0)
    if outlier_features:
        result = cap_outliers_iqr(result, outlier_features)
    if log_features:
        result = log_transform_features(result, log_features)
    return result

In [17]:
log_features = [
        "MonetaryValue", "AOV", "TotalQuantityPurchased",
        "UniqueProducts", "UniqueCategories", "Frequency",
        "AverageDaysBetweenPurchases"
    ]
processed = prepare_customer_features(
        customer_features,
        log_features=log_features,
        outlier_features=model_features,
    )

In [18]:
processed

,CustomerID,Frequency,MonetaryValue,Recency,TotalQuantityPurchased,UniqueProducts,UniqueCategories,AOV,AverageDaysBetweenPurchases
0,CUST_0001,1.609438,5.162383,24,2.197225,1.609438,1.098612,3.793127,4.844187
1,CUST_0002,1.386294,5.966172,5,2.484907,1.386294,1.098612,4.872675,5.466477
2,CUST_0003,1.609438,6.805723,114,2.564949,1.609438,1.098612,5.422745,3.963983
3,CUST_0004,0.693147,3.390473,321,0.693147,0.693147,0.693147,3.390473,0.000000
4,CUST_0005,1.609438,6.649023,14,2.708050,1.609438,1.504077,5.266607,3.957634
...,...,...,...,...,...,...,...,...,...
430,CUST_0496,1.098612,5.416411,281,1.386294,1.098612,1.098612,4.727698,5.466477
431,CUST_0497,1.098612,4.410978,9,1.609438,1.098612,1.098612,3.729901,3.891820
432,CUST_0498,1.609438,5.061392,32,1.945910,1.609438,1.504077,3.693929,4.927254
433,CUST_0499,0.693147,3.736240,31,0.693147,0.693147,0.693147,3.736240,0.000000


In [19]:
from __future__ import annotations
from typing import List, Tuple
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


In [20]:
def scale_features(X: pd.DataFrame | np.ndarray) -> Tuple[StandardScaler, np.ndarray]:
    """Standardize features using z-score standardization."""
    scaler = StandardScaler()
    return scaler, scaler.fit_transform(X)


In [21]:
scaler, X_scaled = scale_features(processed[model_features])

In [22]:
scaler

StandardScaler()

In [23]:
X_scaled

array([[-0.81682386,  1.34221535, -0.1682317 , ...,  1.36356969,
         0.29987768,  0.98186295],
       [-1.03420695,  0.72300553,  0.6674887 , ...,  0.73930714,
         0.29987768,  1.28168332],
       [ 0.21288551,  1.34221535,  1.54039061, ...,  1.36356969,
         0.29987768,  0.5577787 ],
       ...,
       [-0.72529414,  1.34221535, -0.27323496, ...,  1.36356969,
         1.72558543,  1.02188461],
       [-0.73673535, -1.20043584, -1.65102932, ..., -1.19982945,
        -1.12583007, -1.35207627],
       [-0.3934989 , -0.07529476,  0.0248208 , ..., -0.06550727,
         0.29987768,  0.09127365]], shape=(435, 8))

In [24]:
len(X_scaled)

435

In [25]:
def calculate_diff_k_scores(
    X_scaled: np.ndarray,
    k_values: List[int],
    random_state: int = 42,
) -> pd.DataFrame:
    """Calculate K-Means inertia and silhouette scores."""
    if len(X_scaled) < 3:
        raise ValueError("At least three observations are required.")
    records = []
    for k in k_values:
        if k < 2 or k >= len(X_scaled):
            continue
        model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        labels = model.fit_predict(X_scaled)
        records.append({
            "K": k,
            "Inertia": model.inertia_,
            "SilhouetteScore": silhouette_score(X_scaled, labels) if len(np.unique(labels)) > 1 else np.nan,
        })
    return pd.DataFrame(records)

In [26]:
def choose_best_k(elbow_results: pd.DataFrame) -> int:
    """Select K with the highest valid silhouette score."""
    valid = elbow_results.dropna(subset=["SilhouetteScore"])
    if valid.empty:
        return 4
    return int(valid.loc[valid["SilhouetteScore"].idxmax(), "K"])


In [27]:
k_values = list(range(2, min(10, len(processed) - 1) + 1))
elbow = calculate_diff_k_scores(X_scaled, k_values)

best_k = choose_best_k(elbow)
print(f"Selected K: {best_k}")

Selected K: 2


In [28]:
elbow

,K,Inertia,SilhouetteScore
0,2,1855.666741,0.416760
1,3,1416.839335,0.320504
2,4,1173.082869,0.304602
3,5,981.541334,0.292600
4,6,888.037029,0.287691
5,7,816.834377,0.271429
6,8,739.028422,0.277025
7,9,681.531820,0.281239
8,10,630.794673,0.266423


In [29]:
def train_kmeans(
    X_scaled: np.ndarray,
    n_clusters: int,
    random_state: int = 42,
) -> Tuple[KMeans, np.ndarray]:
    """Train the final K-Means model."""
    if n_clusters < 2 or n_clusters >= len(X_scaled):
        raise ValueError("Invalid number of clusters.")
    model = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=50,
        max_iter=500,
    )
    return model, model.fit_predict(X_scaled)

In [30]:
print("[7/10] Training K-Means...")
kmeans, labels = train_kmeans(X_scaled, best_k)
processed["Cluster"] = labels

[7/10] Training K-Means...


In [31]:
labels

array([1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0,
       0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0,

In [32]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score

def _safe_clustering_metrics(X: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    """Calculate clustering metrics while handling DBSCAN noise."""
    labels = np.asarray(labels)
    valid_mask = labels != -1
    if valid_mask.sum() < 3:
        return {
            "SilhouetteScore": np.nan,
            "DaviesBouldinScore": np.nan,
            "NumberOfClusters": 0,
            "NoisePercentage": 100.0,
        }
    valid_labels = labels[valid_mask]
    unique = np.unique(valid_labels)
    if len(unique) < 2:
        return {
            "SilhouetteScore": np.nan,
            "DaviesBouldinScore": np.nan,
            "NumberOfClusters": len(unique),
            "NoisePercentage": float((~valid_mask).mean() * 100),
        }
    return {
        "SilhouetteScore": float(silhouette_score(X[valid_mask], valid_labels)),
        "DaviesBouldinScore": float(davies_bouldin_score(X[valid_mask], valid_labels)),
        "NumberOfClusters": int(len(unique)),
        "NoisePercentage": float((~valid_mask).mean() * 100),
    }


In [33]:
def compare_clustering_models(
    X_scaled: np.ndarray,
    kmeans_clusters: int = 4,
    dbscan_eps: float = 0.8,
    dbscan_min_samples: int = 8,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    """Compare K-Means, Agglomerative and DBSCAN."""
    models = {
        "K-Means": KMeans(n_clusters=kmeans_clusters, random_state=random_state, n_init=20),
        "Agglomerative": AgglomerativeClustering(n_clusters=kmeans_clusters, linkage="ward"),
        "DBSCAN": DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples),
    }
    labels_dictionary = {}
    records = []
    for name, model in models.items():
        labels = model.fit_predict(X_scaled)
        labels_dictionary[name] = labels
        metrics = _safe_clustering_metrics(X_scaled, labels)
        metrics["Model"] = name
        records.append(metrics)
    results = pd.DataFrame(records)
    return results[[
        "Model", "SilhouetteScore", "DaviesBouldinScore",
        "NumberOfClusters", "NoisePercentage"
    ]], labels_dictionary


In [34]:
comparison, _ = compare_clustering_models(X_scaled, best_k)


In [35]:
comparison

,Model,SilhouetteScore,DaviesBouldinScore,NumberOfClusters,NoisePercentage
0,K-Means,0.416760,0.970474,2,0.000000
1,Agglomerative,0.423337,0.934224,2,0.000000
2,DBSCAN,0.322025,0.971811,13,51.264368


In [36]:
def fit_pca(X_scaled: np.ndarray, n_components: int = 3) -> Tuple[PCA, np.ndarray]:
    """Fit PCA for dimensionality reduction."""
    max_components = min(X_scaled.shape[0], X_scaled.shape[1])
    n_components = min(n_components, max_components)
    if n_components < 2:
        raise ValueError("PCA requires at least two components.")
    pca = PCA(n_components=n_components, random_state=42)
    return pca, pca.fit_transform(X_scaled)


In [37]:
pca, X_pca = fit_pca(X_scaled, 3)

In [38]:
pca.explained_variance_ratio_.tolist()

[0.6284581498803876, 0.18972160665083063, 0.10214452817072552]

In [39]:
X_pca

array([[ 1.7145677 , -1.73750985, -0.60205292],
       [ 2.13418967, -0.0104781 , -0.65019028],
       [ 2.66494933,  0.78203901,  0.83057306],
       ...,
       [ 2.02176416, -2.32782027, -0.41613405],
       [-3.37975711, -0.70327437, -1.48679019],
       [ 0.15343927, -0.03637514, -0.36214677]], shape=(435, 3))

In [40]:
def add_pca_coordinates(customer_features: pd.DataFrame, X_pca: np.ndarray) -> pd.DataFrame:
    """Append PCA coordinates to customer records."""
    if len(customer_features) != len(X_pca):
        raise ValueError("Customer rows and PCA rows must have equal length.")
    result = customer_features.copy()
    result["PC1"], result["PC2"] = X_pca[:, 0], X_pca[:, 1]
    if X_pca.shape[1] >= 3:
        result["PC3"] = X_pca[:, 2]
    return result

In [41]:
processed = add_pca_coordinates(processed, X_pca)

In [42]:
processed # final clustered data

,CustomerID,Frequency,MonetaryValue,Recency,TotalQuantityPurchased,UniqueProducts,UniqueCategories,AOV,AverageDaysBetweenPurchases,Cluster,PC1,PC2,PC3
0,CUST_0001,1.609438,5.162383,24,2.197225,1.609438,1.098612,3.793127,4.844187,1,1.714568,-1.737510,-0.602053
1,CUST_0002,1.386294,5.966172,5,2.484907,1.386294,1.098612,4.872675,5.466477,1,2.134190,-0.010478,-0.650190
2,CUST_0003,1.609438,6.805723,114,2.564949,1.609438,1.098612,5.422745,3.963983,1,2.664949,0.782039,0.830573
3,CUST_0004,0.693147,3.390473,321,0.693147,0.693147,0.693147,3.390473,0.000000,0,-4.295461,-1.386494,1.658739
4,CUST_0005,1.609438,6.649023,14,2.708050,1.609438,1.504077,5.266607,3.957634,1,3.450081,0.322232,-0.168063
...,...,...,...,...,...,...,...,...,...,...,...,...,...
430,CUST_0496,1.098612,5.416411,281,1.386294,1.098612,1.098612,4.727698,5.466477,1,-0.197452,-0.562510,2.106208
431,CUST_0497,1.098612,4.410978,9,1.609438,1.098612,1.098612,3.729901,3.891820,1,-0.302927,-1.519785,-1.178765
432,CUST_0498,1.609438,5.061392,32,1.945910,1.609438,1.504077,3.693929,4.927254,1,2.021764,-2.327820,-0.416134
433,CUST_0499,0.693147,3.736240,31,0.693147,0.693147,0.693147,3.736240,0.000000,0,-3.379757,-0.703274,-1.486790


In [43]:
def cluster_profile(customer_features: pd.DataFrame, cluster_column: str = "Cluster") -> pd.DataFrame:
    """Create an aggregated profile for every cluster."""
    if cluster_column not in customer_features.columns:
        raise ValueError(f"Column '{cluster_column}' does not exist.")
    grouped = customer_features.groupby(cluster_column)
    profile = grouped.agg(
        CustomerCount=("CustomerID", "count"),
        AverageRecency=("Recency", "mean"),
        AverageFrequency=("Frequency", "mean"),
        AverageMonetaryValue=("MonetaryValue", "mean"),
        AverageAOV=("AOV", "mean"),
        AverageQuantity=("TotalQuantityPurchased", "mean"),
        AverageUniqueProducts=("UniqueProducts", "mean"),
        AverageUniqueCategories=("UniqueCategories", "mean"),
        AveragePurchaseGap=("AverageDaysBetweenPurchases", "mean"),
    ).reset_index()
    profile["Percentage"] = profile["CustomerCount"] / profile["CustomerCount"].sum() * 100
    return profile.sort_values("AverageMonetaryValue", ascending=False).reset_index(drop=True)


In [44]:
def map_segment_label(profile_row: pd.Series) -> str:
    """Map relative cluster behavior to a business label."""
    profiles = profile_row.attrs.get("all_profiles")
    if profiles is not None and not profiles.empty:
        r_med = profiles["AverageRecency"].median()
        f_med = profiles["AverageFrequency"].median()
        m_med = profiles["AverageMonetaryValue"].median()
    else:
        r_med = profile_row["AverageRecency"]
        f_med = profile_row["AverageFrequency"]
        m_med = profile_row["AverageMonetaryValue"]

    r = profile_row["AverageRecency"]
    f = profile_row["AverageFrequency"]
    m = profile_row["AverageMonetaryValue"]

    if m >= m_med and f >= f_med and r <= r_med:
        return "High-Value Loyal Customers"
    if r > r_med and m >= m_med:
        return "At-Risk Customers"
    if f < f_med and m < m_med:
        return "Low-Engagement Customers"
    if m >= m_med and f < f_med:
        return "High-Spend Occasional Customers"
    return "Regular Customers"

In [45]:
def create_segment_labels(profile: pd.DataFrame) -> pd.DataFrame:
    """Assign business labels to cluster profiles."""
    result = profile.copy()
    labels = []
    for _, row in result.iterrows():
        row.attrs["all_profiles"] = result
        labels.append(map_segment_label(row))
    result["SegmentLabel"] = labels
    return result


In [46]:
profile = create_segment_labels(cluster_profile(processed))

In [47]:
profile

,Cluster,CustomerCount,AverageRecency,AverageFrequency,AverageMonetaryValue,AverageAOV,AverageQuantity,AverageUniqueProducts,AverageUniqueCategories,AveragePurchaseGap,Percentage,SegmentLabel
0,1,281,72.000000,1.339742,5.772811,4.768049,2.148065,1.335429,1.177258,4.141398,64.597701,High-Value Loyal Customers
1,0,154,138.077922,0.735273,4.505594,4.427025,1.263589,0.732641,0.714210,0.370151,35.402299,Low-Engagement Customers


In [48]:
processed

,CustomerID,Frequency,MonetaryValue,Recency,TotalQuantityPurchased,UniqueProducts,UniqueCategories,AOV,AverageDaysBetweenPurchases,Cluster,PC1,PC2,PC3
0,CUST_0001,1.609438,5.162383,24,2.197225,1.609438,1.098612,3.793127,4.844187,1,1.714568,-1.737510,-0.602053
1,CUST_0002,1.386294,5.966172,5,2.484907,1.386294,1.098612,4.872675,5.466477,1,2.134190,-0.010478,-0.650190
2,CUST_0003,1.609438,6.805723,114,2.564949,1.609438,1.098612,5.422745,3.963983,1,2.664949,0.782039,0.830573
3,CUST_0004,0.693147,3.390473,321,0.693147,0.693147,0.693147,3.390473,0.000000,0,-4.295461,-1.386494,1.658739
4,CUST_0005,1.609438,6.649023,14,2.708050,1.609438,1.504077,5.266607,3.957634,1,3.450081,0.322232,-0.168063
...,...,...,...,...,...,...,...,...,...,...,...,...,...
430,CUST_0496,1.098612,5.416411,281,1.386294,1.098612,1.098612,4.727698,5.466477,1,-0.197452,-0.562510,2.106208
431,CUST_0497,1.098612,4.410978,9,1.609438,1.098612,1.098612,3.729901,3.891820,1,-0.302927,-1.519785,-1.178765
432,CUST_0498,1.609438,5.061392,32,1.945910,1.609438,1.504077,3.693929,4.927254,1,2.021764,-2.327820,-0.416134
433,CUST_0499,0.693147,3.736240,31,0.693147,0.693147,0.693147,3.736240,0.000000,0,-3.379757,-0.703274,-1.486790


In [49]:
from __future__ import annotations
from pathlib import Path
import sys
import numpy as np 
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st 
import matplotlib.pyplot as plt


In [50]:
x = profile.sort_values('CustomerCount').copy()
x['Segment'] = 'Cluster' + x.Cluster.astype(str)+' — '+x.SegmentLabel
x

,Cluster,CustomerCount,AverageRecency,AverageFrequency,AverageMonetaryValue,AverageAOV,AverageQuantity,AverageUniqueProducts,AverageUniqueCategories,AveragePurchaseGap,Percentage,SegmentLabel,Segment
1,0,154,138.077922,0.735273,4.505594,4.427025,1.263589,0.732641,0.714210,0.370151,35.402299,Low-Engagement Customers,Cluster0 — Low-Engagement Customers
0,1,281,72.000000,1.339742,5.772811,4.768049,2.148065,1.335429,1.177258,4.141398,64.597701,High-Value Loyal Customers,Cluster1 — High-Value Loyal Customers


In [51]:
fig = px.bar(x,x='CustomerCount',y='Segment',orientation='h',color='SegmentLabel',
                 text='CustomerCount',color_discrete_sequence=px.colors.qualitative.Bold,
                 title='Customer Distribution Across Behavioral Segments',labels={'CustomerCount':'Customers','Segment':''})
fig.update_traces(textposition = 'outside')
fig.update_layout(height=480,showlegend=False,plot_bgcolor='rgba(0,0,0,0)',paper_bgcolor = 'rgba(0,0,0,0)')
fig.show()


In [52]:
v = profile.sort_values('AverageMonetaryValue',ascending=False).copy() 
v['Segment']='Cluster '+v.Cluster.astype(str)+' — '+v.SegmentLabel

In [53]:
v

,Cluster,CustomerCount,AverageRecency,AverageFrequency,AverageMonetaryValue,AverageAOV,AverageQuantity,AverageUniqueProducts,AverageUniqueCategories,AveragePurchaseGap,Percentage,SegmentLabel,Segment
0,1,281,72.000000,1.339742,5.772811,4.768049,2.148065,1.335429,1.177258,4.141398,64.597701,High-Value Loyal Customers,Cluster 1 — High-Value Loyal Customers
1,0,154,138.077922,0.735273,4.505594,4.427025,1.263589,0.732641,0.714210,0.370151,35.402299,Low-Engagement Customers,Cluster 0 — Low-Engagement Customers


In [54]:
fig=px.bar(v,x='Segment',y='AverageMonetaryValue',color='SegmentLabel',text_auto='.2s',
color_discrete_sequence=px.colors.qualitative.Set2,
title='Average Customer Value by Segment',labels={'AverageMonetaryValue':'Average Customer Value (₹)','Segment':''})

In [55]:
largest=profile.loc[profile.CustomerCount.idxmax()]
highest=profile.loc[profile.AverageMonetaryValue.idxmax()]